# MAP01 · Bus and compute selection maps

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/11_maps_review/MAP01_bus_and_compute_selection_maps.ipynb)

**Track:** Maps and review · **Skill axis (sheet):** Maps and review · **Time:** about 50 min

**Skill Hub concepts:** Bus selection map; Compute selection map

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("MAP01")

## Map: pick the link and the processor from the requirements

**Payload throughput** (the bit rate is not the byte rate):

| Bus | Bits on the wire per payload byte | Notes |
|---|---|---|
| UART 8N1 | 10 (start + 8 data + stop) | point to point; RS-485 makes it multi-drop |
| I2C | 9 (8 data + acknowledge), plus address and start/stop per transfer | on-board, 100 kHz to 1 MHz, open drain |
| SPI | 8 | on-board, tens of MHz, one chip select per device |
| Classic CAN | 8 data bytes in about 111 bits (standard frame, no stuffing, with interframe space) | multi-drop, priorities, error confinement |

**Bus decision table** (apply the first rule that matches):

| # | Condition | Choose | Why |
|---|---|---|---|
| 1 | `same_chip` (cores on one system on chip) | RPMsg | shared memory plus mailbox interrupts, no wires |
| 2 | `on_board` and `rate_Bps` <= 50,000 | I2C | two wires, many slow sensors, EEPROMs |
| 3 | `on_board` | SPI | fast ADCs, IMUs, displays |
| 4 | `smart_servos` | Serial | daisy-chained half-duplex servo bus (Dynamixel style) |
| 5 | `deterministic` and `cycle_us` < 1000 and `nodes` > 8 | EtherCAT | many drives, sub-millisecond synchronized cycle |
| 6 | `deterministic` and `rate_Bps` <= 500,000 | CAN FD | robust multi-drop control traffic |
| 7 | otherwise | Ethernet | cameras, lidar, bulk data, the robot computer |

**Compute decision table** (first match):

| # | Condition | Choose | Why |
|---|---|---|---|
| 1 | `custom_io` and `deadline_us` < 100 | FPGA | parallel logic, nanosecond determinism, custom interfaces |
| 2 | `deadline_us` < 1000 and `model_gops` < 0.1 | MCU | interrupts with bounded latency, low power |
| 3 | `signal_processing` and `power_w` < 2 | DSP | multiply-accumulate pipelines for filters and FFTs |
| 4 | `model_gops` >= 1 and `datacenter` | TPU | large-scale training and batch inference |
| 5 | `model_gops` >= 1 and `power_w` < 15 and `int8_ok` | NPU | efficient int8 inference at the edge |
| 6 | `model_gops` >= 1 | GPU | float16 models, flexible kernels, CUDA ecosystem |
| 7 | otherwise | CPU | orchestration, planning, ROS 2 nodes |

In [ ]:
import numpy as np

## Task 1: payload bytes per second
`bus` is one of `"uart"`, `"i2c"`, `"spi"`, `"can"`. Ignore I2C addressing overhead.

In [ ]:
def payload_Bps(bus, bitrate):
    raise NotImplementedError  # TODO

lab.check("throughput", payload_Bps)

## Task 2: encode the two decision tables
Requirements arrive as dictionaries; a missing key means False or 0 (use `req.get(key, default)`).
Return exactly the names in the tables: `"RPMsg"`, `"I2C"`, `"SPI"`, `"Serial"`, `"EtherCAT"`, `"CAN FD"`,
`"Ethernet"` and `"FPGA"`, `"MCU"`, `"DSP"`, `"TPU"`, `"NPU"`, `"GPU"`, `"CPU"`.

In [ ]:
def choose_bus(req):
    raise NotImplementedError  # TODO

def choose_compute(req):
    raise NotImplementedError  # TODO

lab.check("selection", choose_bus, choose_compute)

## Checkpoint: Math
**M1 (Apply).** I2C at 400 kHz: payload bytes per second (9 bits per byte, no addressing)?

**M2 (Apply).** SPI at 20 MHz: bytes per second?

**M3 (Apply).** UART at 1 Mbaud, 8N1: bytes per second?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** Twelve servo drives must update together every 250 microseconds over 15 m of cable. Which bus?
- a) CAN FD
- b) EtherCAT
- c) I2C with a long-distance buffer

**K2 (Create).** Sketch the buses and processors for a 6-axis arm with a wrist camera, joint encoders, a force-torque
sensor and an object detector, and justify each link with the tables (rate, determinism, distance, power).

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `map01.h`).
3. In `roboprep/checks/map01.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()